In [1]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
import numpy as np

# Cargar el modelo VGG16 preentrenado
model = VGG16(weights='imagenet')

# Cargar una imagen de ejemplo
img_path = 'Samoyedo.jpg'
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

# Predicción de la imagen
predictions = model.predict(img_array)

# Decodificar las predicciones
print('Predicted:', decode_predictions(predictions, top=3)[0])

2025-03-24 16:57:17.609445: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-24 16:57:39.389385: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.


35363/35363 [==============================] - 0s 1us/step
Predicted: [('n02114548', 'white_wolf', 0.7118048), ('n02111889', 'Samoyed', 0.17622875), ('n02109961', 'Eskimo_dog', 0.032471023)]


In [2]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
# Cargar el modelo VGG16 preentrenado sin la capa superior
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
# Congelar las capas del modelo base para no entrenarlas
for layer in base_model.layers:
    layer.trainable = False
# Añadir nuevas capas para la nueva tarea
x = Flatten()(base_model.output)
x = Dense(1024, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)
# Asumiendo 10 clases
# Definir el nuevo modelo
model = Model(inputs=base_model.input, outputs=predictions)
# Compilar el modelo
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])



58889256/58889256 [==============================] - 7s 0us/step


In [ ]:
from datasets import load_dataset
dataset = load_dataset("imagenet-1k")
train_dataset = dataset["train"]
from tensorflow.keras.applications import ResNet50
model = ResNet50(weights='imagenet')

In [5]:
# Cargar y preprocesar los datos
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    'data/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)
validation_datagen = ImageDataGenerator(rescale=1./255)
validation_generator = validation_datagen.flow_from_directory(
    'data/validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)
# Entrenar el modelo
model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=10
)
# Guardar el modelo
model.save('vgg16_finetuned.h5')


Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.


ValueError: Asked to retrieve element 0, but the Sequence has length 0

In [ ]:
# Cargar el modelo guardado
from tensorflow.keras.models import load_model
model = load_model('vgg16_finetuned.h5')
# Cargar una nueva imagen para hacer predicciones
img_path = 'nueva_imagen.jpg'
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)
# Hacer predicciones
predictions = model.predict(img_array)
# Decodificar las predicciones
predicted_classes = np.argmax(predictions, axis=1)
# Mostrar la clase predicha
class_indices = train_generator.class_indices
class_indices = {v: k for k, v in class_indices.items()}
predicted_class = class_indices[predicted_classes[0]]
print('Predicted class:', predicted_class)
# Evaluar el modelo
loss, accuracy = model.evaluate(validation_generator)
print('Validation Loss:', loss)
print('Validation Accuracy:', accuracy)